# Notebook 05 — Probability, Cross-Entropy, and Gradients

    ## Learning objectives

    - Compute stable softmax and negative log-likelihood
- Derive the gradient of cross-entropy with respect to logits
- Connect perplexity, calibration, and optimization

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 5.1 Softmax and numerical stability

For logits \(z\), \(p_i=\exp(z_i)/\sum_j\exp(z_j)\). Subtracting
\(\max(z)\) changes neither probabilities nor ratios and prevents overflow.
Temperature \(\tau\) uses `softmax(z / τ)`: lower values sharpen; higher values flatten.


In [ ]:
import torch

def stable_softmax(z, dim=-1):
    shifted = z - z.max(dim=dim, keepdim=True).values
    exp = shifted.exp()
    return exp / exp.sum(dim=dim, keepdim=True)

logits = torch.tensor([1000.0, 1001.0, 999.0])
for temperature in [0.5, 1.0, 2.0]:
    print(temperature, stable_softmax(logits / temperature))


## 5.2 Cross-entropy

For one-hot target \(y\), loss is \(L=-\sum_i y_i\log p_i=-\log p_y\).
The useful simplification is

\[
\frac{\partial L}{\partial z_i}=p_i-y_i.
\]

Incorrect classes receive positive gradients and the target receives a negative
gradient. Mean token loss weights every unmasked token equally unless you intervene.
Perplexity is \(\exp(\text{mean NLL})\), interpretable as an effective branching factor,
but only comparable under the same tokenizer and evaluation protocol.


In [ ]:
logits = torch.tensor([[2.0, 0.5, -1.0]], requires_grad=True)
target = torch.tensor([0])
loss = torch.nn.functional.cross_entropy(logits, target)
loss.backward()
probs = logits.detach().softmax(-1)
expected = probs - torch.nn.functional.one_hot(target, 3)
print("loss:", loss.item(), "perplexity:", loss.exp().item())
print("autograd:", logits.grad)
print("p - y:   ", expected)


## 5.3 Optimization is not evaluation

Lower held-out token loss usually signals better next-token prediction, but product
quality may depend on instruction following, factuality, tool use, safety, or retrieval.
Track training loss for optimization and task-level metrics for decisions. Calibration
asks whether stated probabilities match empirical frequencies; temperature scaling can
improve calibration without changing class ranking.


## 5.4 Log-sum-exp, token reduction, and masking

Stable cross-entropy is normally implemented without constructing probabilities explicitly:

\[
\log\sum_j e^{z_j}=m+\log\sum_j e^{z_j-m},\quad m=\max_j z_j.
\]

The per-token negative log-likelihood is `logsumexp(logits) - target_logit`. Framework
functions fuse these operations for stability and speed. Reduction deserves attention.
A mean over non-padding tokens weights long examples more heavily than short examples. A
mean of per-example means weights examples equally. Neither is universally correct; choose
the unit implied by the task and report it.

Masked loss is not equivalent to deleting input. Masked prompt tokens still condition the
assistant response; they simply receive no direct target loss. Label smoothing replaces a
one-hot target with a mixture containing a small uniform component, discouraging extreme
confidence but potentially harming exact generation. Class weights address imbalance in
classification, while token-level language modeling usually needs data sampling or explicit
token/example weighting instead.


In [ ]:
# Reconstruct cross-entropy from log-sum-exp and compare reductions.
import torch
logits = torch.tensor([
    [[3., 1., 0.], [0., 2., 1.], [1., 0., 3.]],
    [[2., 0., 1.], [1., 3., 0.], [0., 0., 0.]],
])
labels = torch.tensor([[0, 1, 2], [2, 1, -100]])
safe = labels.clamp_min(0)
target_logits = logits.gather(-1, safe.unsqueeze(-1)).squeeze(-1)
nll = torch.logsumexp(logits, -1) - target_logits
valid = labels.ne(-100)
token_mean = nll[valid].mean()
example_means = [(row[mask]).mean() for row, mask in zip(nll, valid)]
example_mean = torch.stack(example_means).mean()
builtin = torch.nn.functional.cross_entropy(logits.flatten(0, 1), labels.flatten())
print("token mean:", token_mean.item(), "builtin:", builtin.item())
print("equal-example mean:", example_mean.item())


## 5.5 Backpropagation beyond the final logits

The elegant gradient `p - y` is only the start. The chain rule propagates it through the
LM head, residual stream, attention, MLPs, embeddings, and earlier tokens. Shared/tied input
embeddings and output weights receive gradient from both roles. Residual connections sum
gradient paths. Normalization changes scale and coupling. Attention routes gradients across
visible prior positions, while the causal mask blocks future paths.

Gradient magnitude alone is not parameter importance. Adaptive optimizers rescale updates
using running moments; weight decay adds a separate shrinkage; mixed precision may scale the
loss before backward. For diagnosis, track global and per-module gradient norms, zero or NaN
fractions, update-to-weight ratios, and activation statistics. Clip after unscaling and before
the optimizer step. Persistent clipping means the learning rate, data, or loss scale deserves
investigation—not that clipping has “fixed” training.


In [ ]:
# Watch a tiny model learn and inspect gradient/update scales.
torch.manual_seed(7)
layer = torch.nn.Linear(4, 3)
opt = torch.optim.SGD(layer.parameters(), lr=0.2)
x = torch.tensor([[1., 0., 1., 0.], [0., 1., 0., 1.]])
y = torch.tensor([0, 2])
for step in range(6):
    before = layer.weight.detach().clone()
    opt.zero_grad(set_to_none=True)
    loss = torch.nn.functional.cross_entropy(layer(x), y)
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(layer.parameters(), 10.0)
    opt.step()
    update = (layer.weight.detach() - before).norm()
    print(step, f"loss={loss.item():.4f}", f"grad={grad_norm:.4f}", f"update={update:.4f}")


## 5.6 Perplexity, entropy, and calibration reference

Perplexity exponentiates average NLL. It is sensitive to tokenizer, domain, sequence
boundary policy, masking, and reduction. It is useful for comparing checkpoints under one
controlled protocol, but a lower perplexity does not guarantee better instruction following,
truthfulness, safety, or downstream utility. Bits per byte/character can make comparisons
across tokenizers fairer by normalizing to the underlying representation.

Entropy describes uncertainty in the model distribution; cross-entropy measures how that
distribution scores observed targets; KL divergence measures distribution discrepancy.
Calibration asks whether events predicted with probability 0.8 happen about 80% of the time.
Generative calibration is difficult because there are many valid sequences. For constrained
labels, use reliability diagrams, expected calibration error, Brier score, and temperature
scaling on held-out data. Never tune calibration on the final test set.


## 5.7 Mathematical reference

| Quantity | Definition | Interpretation |
|---|---|---|
| Softmax | \(p_i=e^{z_i}/\sum_j e^{z_j}\) | Categorical distribution from logits |
| NLL | \(-\log p_y\) | Surprise assigned to observed target |
| Cross-entropy | \(H(q,p)=-\sum q\log p\) | Expected NLL under target distribution |
| Entropy | \(H(p)=-\sum p\log p\) | Distribution uncertainty |
| KL | \(D_{KL}(q\|p)=H(q,p)-H(q)\) | Asymmetric discrepancy |
| Perplexity | \(e^{\text{mean NLL}}\) | Effective branching factor under one protocol |

Use natural logs for nats; divide by `ln(2)` for bits. A probability of zero on an observed event
has infinite NLL, motivating stable finite computations. Averaging logits before loss is not the same
as averaging losses. Never softmax before `cross_entropy`, which expects logits. A lower batch loss
can reflect more padding masked, easier/shorter data, or a changed tokenizer rather than learning.

For gradient checks, compare autograd with finite differences on a tiny float64 model. For training
diagnostics, distinguish loss scale, gradient norm, and actual parameter update after optimizer
adaptation. Report evaluation reduction and denominator explicitly.


## 5.7 Deriving the softmax-cross-entropy gradient

For logits `z`, probabilities `p=softmax(z)`, and one-hot target `y`, cross-entropy has gradient `p-y`. The result explains why incorrect confident predictions receive large corrective updates and correct confident predictions receive small ones. Batch and token reductions multiply this expression by their denominator, while ignored labels contribute zero. Derive it once using the softmax Jacobian, then confirm it with autograd and finite differences. This compact identity is the starting point for understanding label smoothing, temperature, distillation KL losses, and why applying a second softmax before framework cross-entropy damages the objective.


In [ ]:
z=torch.tensor([1.2,-.3,.7],dtype=torch.float64,requires_grad=True); target=torch.tensor([2])
loss=torch.nn.functional.cross_entropy(z[None],target); loss.backward()
expected=torch.softmax(z.detach(),0); expected[target]-=1
print(loss.item(),z.grad,expected); torch.testing.assert_close(z.grad,expected)


## 5.8 Gradient flow through a two-layer network

Matrix calculus becomes practical when every gradient is associated with a shape. For `h=xW1`, activation `a=ReLU(h)`, and logits `aW2`, backpropagation first computes the logits gradient, forms outer products for `W2`, masks the hidden gradient where ReLU was inactive, and forms outer products for `W1`. Work a one-example network manually and compare every tensor with autograd. Saturated or inactive units, scale changes, and reduction factors become visible. This habit makes later transformer residual, normalization, and attention gradients less mysterious even though production code delegates differentiation to PyTorch.


In [ ]:
x=torch.tensor([[1.,-2.]],requires_grad=False); W1=torch.tensor([[.5,-.4],[.3,.2]],requires_grad=True); W2=torch.tensor([[.7,-.1],[-.2,.8]],requires_grad=True)
h=x@W1; a=h.relu(); logits=a@W2; loss=-torch.log_softmax(logits,-1)[0,1]; loss.backward()
g=torch.softmax(logits.detach(),-1); g[0,1]-=1; manual_W2=a.detach().T@g; manual_W1=x.T@((g@W2.detach().T)*(h.detach()>0))
torch.testing.assert_close(W2.grad,manual_W2); torch.testing.assert_close(W1.grad,manual_W1); print(W1.grad,W2.grad)


## Exercises

    1. Derive the binary cross-entropy gradient from first principles.
2. Plot entropy as temperature ranges from 0.1 to 3.
3. Construct two tokenizers for which identical text produces incomparable perplexities.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
